# AI 데이터센터 열섬 — 요약 결과

> **이 노트북은 원자료 없이 돌아간다.** 개인정보가 든 질병관리청 원자료도, 36GB 위성영상도 필요 없다.
> 저장소의 `DERIVED_*.csv` 집계본만 있으면 아래 숫자가 그대로 재현된다.
>
> 전체 분석 과정(167셀, 우리가 틀렸다가 고친 자리 포함)은 `산업전력_검증노트북_2026-07-18.ipynb`에 있다.
> 이 노트북은 **결과만** 선형으로 보여준다. 다만 **한계·가정·범위 표기는 그대로 남겼다** — 그걸 빼면 읽는 사람을 오도한다.

작성 2026-07-28

In [ ]:
import pandas as pd, numpy as np
pd.set_option('display.width', 170)
print('pandas', pd.__version__, '· numpy', np.__version__)

pandas 2.2.3 · numpy 2.2.3


## 1. 부지는 이미 뜨겁다

Landsat 8/9 지표온도(LST) 90여 장면. 각 부지를 주변 10~20km 링과 비교했다.

두 기준을 병기한다 — **전체 기준**(숲·농지 포함)과 **주거지 기준**(링 안에서 주거지 화소만).
후자가 "사람 사는 곳보다 얼마나 뜨거운가"에 맞는 질문이다.

In [ ]:
D14 = pd.read_csv('DERIVED_0722_주거지기준선_ΔT.csv', encoding='utf-8-sig')
g = (D14.groupby('부지')
       .agg(장면수=('site', 'size'), 부지LST=('site', 'mean'),
            전체기준=('base_r45', 'mean'), 주거기준=('base_res_lc', 'mean'))
       .assign(ΔT_전체=lambda d: d.부지LST - d.전체기준,
               ΔT_주거=lambda d: d.부지LST - d.주거기준)
       .sort_values('ΔT_주거', ascending=False).round(1))
print('부지별 지표온도 이상 (℃, 주변 대비)')
print(g[['장면수', 'ΔT_전체', 'ΔT_주거']].to_string())
print()
print(f'→ 전체 기준 {g.ΔT_전체.min():.1f} ~ {g.ΔT_전체.max():.1f}℃ · 주거 기준 {g.ΔT_주거.min():.1f} ~ {g.ΔT_주거.max():.1f}℃')
print('→ 기준을 주거지로 좁혀도 **모든 부지가 여전히 더 뜨겁다**. 순위도 대체로 보존된다.')
print('⚠ 이것은 **지표온도**(위성이 재는 표면 온도)이지 사람이 숨쉬는 높이의 기온이 아니다. 환산은 3절.')

부지별 지표온도 이상 (℃, 주변 대비)
         장면수  ΔT_전체  ΔT_주거
부지                        
포항·제철     13   10.5   10.5
당진1철강      9    9.5   10.3
구미·전자     20    6.4    7.2
심팩 포항     14    6.8    7.2
울산미포      16    4.1    6.1
동해 북평2    21    4.0    4.5
현대제철 인천    9    3.7    4.4
동해북평      22    4.0    4.3
동국제강 인천    9    3.3    3.9
광양·제철     11    3.0    2.9
온산        26    2.5    2.7
여수·석화      7    1.4    1.4

→ 전체 기준 1.4 ~ 10.5℃ · 주거 기준 1.4 ~ 10.5℃
→ 기준을 주거지로 좁혀도 **모든 부지가 여전히 더 뜨겁다**. 순위도 대체로 보존된다.
⚠ 이것은 **지표온도**(위성이 재는 표면 온도)이지 사람이 숨쉬는 높이의 기온이 아니다. 환산은 3절.


## 2. 열은 거리에 따라 식는다 — 형태를 논문 보고점으로 골랐다

케임브리지(Marinoni et al.)가 **본문에 적은 3점**에 세 형태를 맞춰봤다. 임의로 지수형을 고른 것이 아니다.

In [ ]:
CAMB_DT0, CAMB_HALF = 2.07, 4.2
_exp = lambda r: CAMB_DT0 * np.exp(-r * np.log(2) / CAMB_HALF)
_gau = lambda r: CAMB_DT0 * np.exp(-np.log(2) * (r / 4.5) ** 2)


def _lin(r):
    return CAMB_DT0 + (1.0 - CAMB_DT0) * r / 4.5 if r <= 4.5 else max(0.0, (10 - r) / 5.5)


print(f"{'형태':8}{'부지(2.07)':>12}{'4.5km(1.0)':>12}{'7km(30%)':>11}{'10km(>0)':>11}  판정")
for nm, f in [('지수', _exp), ('가우시안', _gau), ('선형', _lin)]:
    ok = (abs(f(0) - 2.07) < .05 and abs(f(4.5) - 1.0) < .12
          and abs(f(7) / CAMB_DT0 - .30) < .06 and f(10) > .05)
    print(f'{nm:8}{f(0):>11.2f}℃{f(4.5):>11.2f}℃{f(7)/CAMB_DT0*100:>10.0f}%{f(10):>10.2f}℃  {"통과" if ok else "탈락"}')
print()
print('→ 가우시안은 7km에서 19%로 너무 빨리 식고(보고값 30%), 선형은 10km에서 0이 되어 "10km까지 도달"과 어긋난다.')
print('→ **지수형만 세 점을 모두 통과**한다. 반감기 4.2km.')

형태          부지(2.07)  4.5km(1.0)   7km(30%)   10km(>0)  판정
지수             2.07℃       0.99℃        31%      0.40℃  통과
가우시안           2.07℃       1.03℃        19%      0.07℃  탈락
선형             2.07℃       1.00℃        26%      0.00℃  탈락

→ 가우시안은 7km에서 19%로 너무 빨리 식고(보고값 30%), 선형은 10km에서 0이 되어 "10km까지 도달"과 어긋난다.
→ **지수형만 세 점을 모두 통과**한다. 반감기 4.2km.


## 3. 지표온도를 기온으로 옮긴다

위성이 재는 것은 **땅 표면**이고 건강에 영향을 주는 것은 **공기**다. 둘을 잇는 계수 β를 우리 관측소 자료로 직접 구했다.

- **채택 β = 0.258** — 장면 고정효과 회귀, AWS 347지점. **직접 측정값**
- 상한 후보 0.459 — 측정오차를 가정해 보정한 값. **민감도가 극단이라 단독으로 쓰지 않는다**

In [ ]:
BETA = 0.258
print(f"{'거리':>8}{'ΔLST':>9}{'Δ기온':>9}")
for r in (0, 0.5, 1, 2, 3, 4.5, 7.25, 10):
    print(f'{r:>6}km{_exp(r):>8.2f}℃{_exp(r)*BETA:>8.2f}℃')
print()
print('→ 부지 경계에서도 기온으로는 **+0.53℃**, 3km에서 +0.33℃, 10km에서 +0.10℃다.')
print('→ 지표온도 두 자릿수와 기온 0.5℃ 미만은 **모순이 아니라 다른 두 측정**이다.')
print('⚠ β는 *서로 다른 장소*를 비교해 얻은 값이다. "한 곳에 열을 부으면 얼마나 오르나"와는 다르다 —')
print('   한국에서 가동 중인 데이터센터 주변을 차량으로 실측하면 이 구멍이 직접 메워진다.')

      거리     ΔLST      Δ기온
     0km    2.07℃    0.53℃
   0.5km    1.91℃    0.49℃
     1km    1.76℃    0.45℃
     2km    1.49℃    0.38℃
     3km    1.26℃    0.33℃
   4.5km    0.99℃    0.25℃
  7.25km    0.63℃    0.16℃
    10km    0.40℃    0.10℃

→ 부지 경계에서도 기온으로는 **+0.53℃**, 3km에서 +0.33℃, 10km에서 +0.10℃다.
→ 지표온도 두 자릿수와 기온 0.5℃ 미만은 **모순이 아니라 다른 두 측정**이다.
⚠ β는 *서로 다른 장소*를 비교해 얻은 값이다. "한 곳에 열을 부으면 얼마나 오르나"와는 다르다 —
   한국에서 가동 중인 데이터센터 주변을 차량으로 실측하면 이 구멍이 직접 메워진다.


## 4. 기온이 1℃ 오르면 온열질환은 몇 배가 되나

부지별 배수는 채택 규칙을 따른다 — 환자 표본 n≥100이고 최근접 관측소가 10km 이내면 **시군구** 값,
아니면 **시도** 값. 표본이 얇은 곳에 얇은 추정을 쓰지 않기 위한 규칙이다.

In [ ]:
BS = pd.read_csv('DERIVED_0725_부지별_dose배수.csv', encoding='utf-8-sig')
print('부지별 1℃당 온열질환 배수')
print(BS.to_string(index=False))
_num = BS.select_dtypes('number')
_col = [c for c in _num.columns if _num[c].between(1.2, 2.0).mean() > .7]
if _col:
    v = _num[_col[0]]
    print()
    print(f'→ 부지별 {v.min():.2f}~{v.max():.2f}배. 전국 상수 하나로 뭉개면 일부 부지는 40%가량 과대해진다.')
print()
print('⚠ 이 배수는 "다른 조건이 같은데 기온만 오를 때"가 아니라 **"더운 날의 총효과"**다.')
print('   강수·일사까지 통제하면 1.51로 내려가지만, AIDC가 그 변수들을 어떻게 바꿀지 우리가 정할 수 없으므로')
print('   통제하지 않은 쪽(총효과)을 채택했다. ⚠ 채택한 쪽이 추정치가 10~13% **큰** 쪽이라는 점을 밝혀둔다.')

부지별 1℃당 온열질환 배수
        시군구  채택배수m  표본n 최근접관측소  관측소거리km
   인천광역시 동구 1.6305   19     인천      1.2
   인천광역시 서구 1.5640  200     인천      9.3
   울산광역시 남구 1.5141   98     울산      2.8
  울산광역시 울주군 1.5654  115     울산     10.0
   경상북도 포항시 1.3681  278     포항      1.4
   경상북도 구미시 1.4629  196     구미      4.5
   충청남도 당진시 1.5846  135     서산     20.9
강원특별자치도 동해시 1.5676   34     동해      1.7
   전라남도 광양시 1.3918  191    광양시      3.3
   전라남도 여수시 1.5449  201     여수      5.3

→ 부지별 1.37~1.63배. 전국 상수 하나로 뭉개면 일부 부지는 40%가량 과대해진다.

⚠ 이 배수는 "다른 조건이 같은데 기온만 오를 때"가 아니라 **"더운 날의 총효과"**다.
   강수·일사까지 통제하면 1.51로 내려가지만, AIDC가 그 변수들을 어떻게 바꿀지 우리가 정할 수 없으므로
   통제하지 않은 쪽(총효과)을 채택했다. ⚠ 채택한 쪽이 추정치가 10~13% **큰** 쪽이라는 점을 밝혀둔다.


## 5. 노출 인구 — 여기가 가장 단단하다

통계청 집계구 실측. 모형도 가정도 없다.

In [ ]:
F = pd.read_csv('DERIVED_0721_산단_반경별_인구_10km.csv', encoding='utf-8-sig')
print(F.to_string(index=False))
print()
print('→ "산단은 외곽이라 사람이 없다"는 **대표좌표 한 점에서 잰 3km 안에서만** 참이다.')
print('   실형상(폴리곤) 경계로 재면 이미 3km 안이 수만~수십만 명이다.')
print('⚠ 포항·여수는 산단 실형상 자료가 없어 대표좌표 폴백을 썼다 — 그 값은 **하한**이다.')
print('⚠ 같은 방식으로 계산했던 광양은 대표좌표가 바다에 찍혀 3km 인구가 73명으로 나왔지만,')
print('   실형상으로 다시 재자 72,511명이었다. 점 기준 값을 단독으로 인용하면 안 된다.')

                   부지  인구_3km  인구_4.5km  인구_10km  고령_10km  고령비%
        동국제강 인천 (폐·점)  206965    509938  2345227   403200  17.2
        현대제철 인천 (폐·점)  187102    485960  2341418   403412  17.2
           울산미포 (폴리곤)  467637    623152   982108   152655  15.5
             온산 (폴리곤)   32241     63572   484535    79845  16.5
          구미·전자 (폴리곤)  166640    248108   458556    50869  11.1
     포항·제철 (점·YUCH결측)    9935     92833   418357    83156  19.9
     여수·석화 (점·YUCH결측)     409      2776   186495    32741  17.6
    동해북평(국가·대조) (폴리곤)   39062     75361   124462    28917  23.2
동해 북평2(GS2.4GW) (폴리곤)   25978     69791   123274    28459  23.1
          광양·제철 (폴리곤)   72511     90220   120229    18943  15.8
          당진1철강 (폴리곤)    4334      9343    67149    10325  15.4

→ "산단은 외곽이라 사람이 없다"는 **대표좌표 한 점에서 잰 3km 안에서만** 참이다.
   실형상(폴리곤) 경계로 재면 이미 3km 안이 수만~수십만 명이다.
⚠ 포항·여수는 산단 실형상 자료가 없어 대표좌표 폴백을 썼다 — 그 값은 **하한**이다.
⚠ 같은 방식으로 계산했던 광양은 대표좌표가 바다에 찍혀 3km 인구가 73명으로 나왔지만,
   실형상으로 다시 재자 72,511명이었다. 점 기준 값을

## 6. 열은 사라지지 않는다 — 기온이 덜 오르면 물이 가져간 것이다

데이터센터가 쓴 전력은 거의 전부 열이 되고, 나갈 길은 셋뿐이다.

1. 공기로 **현열**(드라이쿨러) → 기온 상승
2. 물 증발로 **잠열**(증발식 냉각탑) → 물 소비 + 습도
3. 수역으로 **온배수** → 수온 상승

**닫힌 회계다.** 어느 쪽을 줄여도 다른 쪽이 드러난다.

In [ ]:
LAT = 2.44e6                      # 물 증발잠열 J/kg
GW, W_PER_GW = 2.4, 3.8e4         # 북평2 발표용량 · 물 소비 톤/일/GW  ⚠출처 미확인 추정
W = W_PER_GW * GW
print(f'북평2 2.4GW · 물 소비 제시값 {W:,.0f} 톤/일   ⚠ 이 값은 출처 미확인 추정이다')
print()
print(f"{'PUE':>5}{'총배출열':>10}{'전량증발 필요':>15}{'증발비중':>9}{'공기로 가는 현열':>18}")
for pue in (1.1, 1.2, 1.3, 1.5):
    Q = GW * 1e9 * pue
    need = Q / LAT * 86400 / 1e3
    lat = W * 1e3 / 86400 * LAT
    sen = max(Q - lat, 0.0)
    print(f'{pue:>5.1f}{Q/1e9:>9.2f}GW{need:>13,.0f}톤{W/need*100:>8.0f}%{sen/1e9:>14.2f}GW ({sen/Q*100:.0f}%)')
print()
print('→ 전력 수치와 물 수치는 **서로 다른 출처인데 에너지 보존으로 맞물린다**.')
print('→ 우리 추가온열 추정이 작게 나오는 것은 결함이 아니라 **냉각방식의 귀결**일 수 있다.')
print('⚠ 물 수치가 출처 미확인이므로 **숫자가 아니라 구조**로 읽어야 한다 —')
print('   "셋 중 하나는 반드시 감당한다"는 것은 물 수치가 틀려도 성립한다.')

북평2 2.4GW · 물 소비 제시값 91,200 톤/일   ⚠ 이 값은 출처 미확인 추정이다

  PUE      총배출열        전량증발 필요     증발비중         공기로 가는 현열
  1.1     2.64GW       93,482톤      98%          0.06GW (2%)
  1.2     2.88GW      101,980톤      89%          0.30GW (11%)
  1.3     3.12GW      110,479톤      83%          0.54GW (17%)
  1.5     3.60GW      127,475톤      72%          1.02GW (28%)

→ 전력 수치와 물 수치는 **서로 다른 출처인데 에너지 보존으로 맞물린다**.
→ 우리 추가온열 추정이 작게 나오는 것은 결함이 아니라 **냉각방식의 귀결**일 수 있다.
⚠ 물 수치가 출처 미확인이므로 **숫자가 아니라 구조**로 읽어야 한다 —
   "셋 중 하나는 반드시 감당한다"는 것은 물 수치가 틀려도 성립한다.


## 7. 아직 못 한 것

| | |
|---|---|
| **β의 정체** | 서로 다른 장소를 비교해 얻은 값이다. 열 주입 반응이 아니다 — 한국 실측이 필요하다 |
| **플룸(바람 방향)** | 우리 ΔT는 **등방**이다. 열이 바람 쪽으로 몰리면 dose가 볼록해 추가 환자가 **오히려 늘어난다**. 모형이 오면 꽂을 자리는 본 노트북 §9-j·§12-e에 있다 |
| **케임브리지 ↔ Sailor** | 위성 링평균과 차량 풍하최대를 하나의 기하로 화해시키려다 실패했다. 현실적 플룸 폭에서는 우리 쪽이 1.3~5배 크게 나온다 |
| **포항·여수 경계** | 실형상 자료 없음 → 점 폴백 = 하한 |
| **물** | 열의 대부분이 물로 나간다는 것까지는 계산했지만 실제 취수원·방류 계획은 모른다 |

---

**반박을 환영한다.** 전체 과정과 우리가 틀렸던 자리는 `산업전력_검증노트북_2026-07-18.ipynb` 셀 1에 **자기수정 11건**으로 적어두었다.